# **Lecture 12 Quiz: Generalization & Bias-Variance**

### **Part 1: Conceptual Stress-Tests**

**Question 1: The Training Paradox**

In supervised learning, we optimize our weights $\mathbf{W}$ by minimizing the loss $\mathcal{L}$ strictly on the training data ($D_{Tr}$):

$$\min_{\mathbf{W}} \mathcal{L}(\mathbf{W}, \mathbf{x}) \quad \forall \mathbf{x} \in D_{Tr}$$

**Why does a solution optimized only on training data usually work reasonably well on unseen test data?**

A) Because we use the test data $D_{Te}$ during the training phase to adjust weights.

B) Because of the assumption that training and test data are sampled from the same underlying distribution.

C) It doesn't; it is purely a matter of luck and "magic" that has no mathematical explanation.

D) Because once a model is trained on $D_{Tr}$, it is mathematically guaranteed to work on any data.

#### Answer:
B) Because of the assumption that training and test data are sampled from the same underlying distribution.

explanation : because we assume that both training and test data are drawn from the same underlying distribution optimizing on the training data allows the model to learn patterns that are likely to generalize well to unseen data.


**Question 2: Bias vs. Variance**

You are comparing two models:
* **Model A:** A simple linear line that misses many points but is very stable.
* **Model B:** A high-degree polynomial that hits every single training point but fluctuates wildly with small data changes.



**Which statement is true?**

A) Model A has High Variance; Model B has High Bias.

B) Model A has High Bias (Underfitting); Model B has High Variance (Overfitting).

C) Both models have high Empirical Error.

D) Model B is the "Best Fit" because its training error is zero.

#### Answer:

B) Model A has High Bias (Underfitting); Model B has High Variance (Overfitting).

explanation : model A is underfitting because it is too simple to capture the underlying patterns in the data
model B is overfitting because it is too complex and captures noise in the training data instead of the underlying pattern leading to high variance when applied to new data.



### **Part 2: The Logic of Fit**

**Question 3: Parameters vs. Hyperparameters**

In the coding exercises, you had to choose a **Learning Rate ($\eta$)** and the number of **Epochs**, while the algorithm calculated the **Weights ($\mathbf{w}$)**.

**Categorize these correctly:**
1.  **Weights ($\mathbf{w}$)** are: Parameter
2.  **Learning Rate ($\eta$)** is: Hyperparameter

*Choices: Parameter, Hyperparameter*

explaination : parameter are the internal variables of the model that are learned from the training data whereas
hyperparameters are the external configurations that are set before training and control the learning process.

### **Part 3: Mathematical Reasoning**

**Question 4: Regularization and Eigenvalues**

In Ridge Regression, we often solve the system using the matrix $(XX^T + \lambda I)$. We know that $XX^T$ is Positive Semi-Definite (PSD) with eigenvalues $\lambda_i \ge 0$. By adding the regularization term $\lambda I$, we effectively shift the eigenvalues to $(\lambda_i + \lambda)$.



**How does increasing this regularization parameter ($\lambda$) affect the Bias-Variance tradeoff?**

A) It increases Variance by making the model more sensitive to small changes in $X$.

B) It decreases Bias by allowing the model to fit the training data more closely.

C) It decreases Variance (improving stability) but increases Bias (making the model simpler).

D) It has no mathematical impact on the stability of the matrix inversion.

#### Answer:
C) It decreases Variance (improving stability) but increases Bias (making the model simpler).

explanation : adding the penalty term $\lambda I$ increases the eigenvalues which improves the stability of the matrix inversion and reduces variance. however it also makes the model simpler and less flexible which can increase bias.

### **Part 4: Coding Challenge — Train Validation Split**

In [3]:
import numpy as np

# --- 1. Setup Synthetic Data ---
np.random.seed(42)
X = np.random.randn(100, 2)
y = np.random.randint(0, 2, 100)

# --- 2. The Logic Function ---

def train_test_split(X, y, test_ratio=0.2):
    # 1. Generate indices [0, 1, ..., N-1]
    # TODO: Create an array of indices
    indices = np.arange(X.shape[0])

    # 2. Shuffle indices in-place
    np.random.shuffle(indices)
    
    # 3. Calculate the split point
    # TODO: Calculate split_idx as an integer (80% if ratio is 0.2)
    split_idx = int(X.shape[0] * (1 - test_ratio))
    
    # 4. Partition the indices
    # TODO: Slice indices into train_idx and val_idx
    train_idx = indices[:split_idx] 
    val_idx = indices[split_idx:]
    
    # 5. Slice original data
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    return X_train, X_val, y_train, y_val

In [4]:
# --- 3. The Integrity Testing Suite ---

def run_split_integrity_tests(X, y):
    print("--- Running Data Split Integrity Tests ---")
    
    # Execute split (80% Train, 20% Validation)
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_ratio=0.2)
    
    # Check 1: Totals
    # Does the sum of split samples equal the original count?
    total_samples = X_train.shape[0] + X_val.shape[0]
    count_pass = (total_samples == X.shape[0])
    
    # Check 2: Data Leakage (Overlap)
    # Ensure no training row is present in the validation set.
    train_set = set(map(tuple, X_train))
    val_set = set(map(tuple, X_val))
    overlap = train_set.intersection(val_set)
    leakage_pass = (len(overlap) == 0)
    
    # Check 3: Label-Feature Sync
    # Ensure labels stayed matched with their respective feature rows.
    label_pass = (X_train.shape[0] == y_train.shape[0]) and (X_val.shape[0] == y_val.shape[0])

    print(f"1. Sample Count Integrity: {'✅ Pass' if count_pass else '❌ Fail'}")
    print(f"2. Zero Data Leakage:      {'✅ Pass' if leakage_pass else '❌ Fail'}")
    print(f"3. Label-Feature Sync:     {'✅ Pass' if label_pass else '❌ Fail'}")
    
    return X_train, X_val, y_train, y_val

# --- 4. Execution ---

# Run tests and store the resulting data for modeling
X_train, X_val, y_train, y_val = run_split_integrity_tests(X, y)

print(f"\nFinal Shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")

--- Running Data Split Integrity Tests ---
1. Sample Count Integrity: ✅ Pass
2. Zero Data Leakage:      ✅ Pass
3. Label-Feature Sync:     ✅ Pass

Final Shapes:
X_train: (80, 2), y_train: (80,)
X_val:   (20, 2), y_val:   (20,)
